In [0]:
create catalog if not exists my_catalog;
create database if not exists my_schema;

In [0]:
use catalog my_catalog;
use schema my_schema;

In [0]:
select current_catalog(), current_schema()

In [0]:
-- volumenes es utilizado como raw files to be ingested.
LIST '/Volumes/my_catalog/my_schema/mi_volumen'

Podemos cargar json / parquet /cvs ect

In [0]:
SELECT * FROM json.`/Volumes/my_catalog/my_schema/mi_volumen`;

# 1. Batch: read_files table-valued function


### Use of `read_files` for Batch Processing

The read_files function allows you to efficiently load multiple files (e.g., JSON, Parquet, CSV) in batch mode for processing. [read_files](https://docs.databricks.com/en/ingestion/cloud-object-storage/copy-into/index.html)

nota: A *_rescued_data* column is automatically included by default to capture any data that doen't match the inferred schema.

In [0]:
SELECT * FROM read_files(
        '/Volumes/my_catalog/my_schema/mi_volumen',
        format => 'json'
)
LIMIT 10 ;

read_files with CTAS

In [0]:
drop table if exists historical_users_bronze_ctas_rf;

create table if not exists my_schema.historical_users_bronze_ctas_rf
SELECT * FROM read_files(
        '/Volumes/my_catalog/my_schema/mi_volumen',
        format => 'json'
);

-- Preview the Delta table
SELECT * FROM my_schema.historical_users_bronze_ctas_rf LIMIT 10;


### Delta Table description

In [0]:
describe table extended my_schema.historical_users_bronze_ctas_rf;

### Python Ingestion

In [0]:
%python
## 1. Read the parquet files from the volume into a Spark Dataframe
df = (spark
    .read
    .format("json")
    .load("/Volumes/my_catalog/my_schema/mi_volumen")
    )
display(df)

In [0]:
%python
## 2. Write to the datframe as a Delta table( overwrite the table if it exists)
(df
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("my_catalog.my_schema.historical_users_bronze_python")
)

In [0]:
%python
## 3. Read and view the table
users_bronze_table = spark.table("my_schema.historical_users_bronze_ctas_rf")
display(users_bronze_table)

# 2. Incremental Data Ingestion with COPY INTO

The `COPY INTO` command allows you to efficiently load data from cloud storage or file location into Delta tables. [COPY INTO Documentation](https://docs.databricks.com/en/ingestion/cloud-object-storage/copy-into/index.html)

In [0]:
drop table if exists historical_users_bronze_ci;

create table historical_users_bronze_ci(
  Country STRING,
  Firstname STRING,
  Lastname STRING
);

copy into historical_users_bronze_ci
from '/Volumes/my_catalog/my_schema/mi_volumen'
FILEFORMAT = JSON;

Hoe to fix the missmath? managed = true

In [0]:
copy into historical_users_bronze_ci
from '/Volumes/my_catalog/my_schema/mi_volumen'
FILEFORMAT = JSON
copy_options('mergeSchema'='true');

select * from historical_users_bronze_ci;


Another way

In [0]:
drop table if exists historical_users_bronze_ci_no_schema;

create table historical_users_bronze_ci_no_schema;

copy into historical_users_bronze_ci_no_schema
from '/Volumes/my_catalog/my_schema/mi_volumen'
FILEFORMAT = JSON
copy_options('mergeSchema'='true');


In [0]:
select * from historical_users_bronze_ci_no_schema limit 10;

# 3. Idempotencia: Copy into no carga nada si no ha cambiado. Si se añadiera un nuevo archivo que no ha leído previamente si lo recargaría de manera incremental

In [0]:
copy into historical_users_bronze_ci_no_schema
from '/Volumes/my_catalog/my_schema/mi_volumen'
FILEFORMAT = JSON
copy_options('mergeSchema'='true');


# 4. Streming Tables

In [0]:
LIST '/Volumes/my_catalog/my_schema/my_csv'

In [0]:
%python
schema_name = 'my_schema';

In [0]:

select * from read_files(
        '/Volumes/my_catalog/my_schema/my_csv',
        format => 'csv',
        sep => ',',
        header => true
)

AutoLoader

Streaming Information : [Streaming tables](https://docs.databricks.com/aws/en/dlt/streaming-tables)

In [0]:
create or refresh streaming table sql_csv_autoloader
schedule every 1 week --- Scheduling the refres is optional
as
select * from read_files(
        '/Volumes/my_catalog/my_schema/my_csv',
        format => 'csv',
        sep => ',',        
        header => true
);


In [0]:
select count(*) from my_catalog.my_schema.sql_csv_autoloader;

In [0]:
describe history my_catalog.my_schema.sql_csv_autoloader;

In [0]:
DESCRIBE TABLE EXTENDED my_catalog.my_schema.sql_csv_autoloader;

In [0]:
drop table if exists sql_csv_autoloader;

In [0]:
refresh streaming table sql_csv_autoloader;

# 5. Python

In [0]:
%python
msc="my_schema"
       
spark.sql(f'LIST "/Volumes/my_catalog/{msc}/my_csv"').display()

In [0]:
%python
spark.sql(f'CREATE VOLUME IF NOT EXISTS my_catalog.my_schema.my_csv_python')

In [0]:
%python
msc="my_schema"
       
spark.sql(f'LIST "/Volumes/my_catalog/{msc}/my_csv_python"').display()

In [0]:
%python
# Install the Delta Live Tables library
%pip install databricks-dlt

In [0]:
%python
%restart_python

In [0]:
%python
import dlt

checkpoint_path = "/Volumes/my_catalog/my_schema/my_csv_python"

@dlt.table
def python_csv_autoloader():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", ",")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .load("/Volumes/my_catalog/my_schema/my_csv_python")
    )

@dlt.table
def write_to_delta():
    df = dlt.read_stream("python_csv_autoloader")
    return df.writeStream.format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", checkpoint_path) \
        .toTable("my_catalog.my_schema.python_csv_autoloader")

Creamos la tabla delta

In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Definir el esquema de las columnas
schema = StructType([
    StructField("OnlineRetailer", StringType(), True),
    StructField("SalesMonth", DateType(), True),
    StructField("Title", StringType(), True),
    StructField("Vintage", IntegerType(), True),
    StructField("Variety", StringType(), True),
    StructField("Score", IntegerType(), True),
    StructField("ListPrice", DoubleType(), True),
    StructField("Quantity", IntegerType(), True)
])


In [0]:
CREATE VOLUME my_catalog.my_schema.my_csv_python;

In [0]:
CREATE VOLUME my_catalog.my_schema.checkpoints;

In [0]:
%python
query.stop()

In [0]:
SELECT * FROM my_catalog.my_schema.ventas_limpias;

In [0]:
drop volume if exists my_catalog.my_schema.my_csv_python;

In [0]:
%python
# Verificar la existencia de la ruta en DBFS
dbfs_path = "dbfs:/Volumes/my_catalog/my_schema/my_csv_python/"
files = dbutils.fs.ls(dbfs_path)

# Imprimir los archivos encontrados
for file in files:
    print(file.path)


In [0]:
%python
# Verificar si el cluster está activo
cluster_info = spark.conf.get("spark.databricks.clusterUsageTags.clusterName", "Cluster name not available")
print(cluster_info)


In [0]:
%python
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Define el esquema de las columnas
schema = StructType([
    StructField("OnlineRetailer", StringType(), True),
    StructField("SalesMonth", DateType(), True),
    StructField("Title", StringType(), True),
    StructField("Vintage", IntegerType(), True),
    StructField("Variety", StringType(), True),
    StructField("Score", IntegerType(), True),
    StructField("ListPrice", DoubleType(), True),
    StructField("Quantity", IntegerType(), True)
])

# Ruta al directorio de entrada
input_path = "dbfs:/Volumes/my_catalog/my_schema/my_csv_python/"

# Leer los archivos CSV en streaming desde la ruta especificada
raw_stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(input_path)

# Transformación de los datos
transformed_stream_df = raw_stream_df \
    .withColumn("SalesMonth", to_date(col("SalesMonth"), "yyyy-MM-dd")) 
    # .dropna(subset=["OnlineRetailer", "SalesMonth", "Quantity", "ListPrice"])

# Escribir el DataFrame transformado a la tabla Delta
output_table = "my_catalog.my_schema.ventas_limpias"

# Configuración del streaming para escribir en Delta
query = transformed_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "dbfs:/Volumes/my_catalog/my_schema/checkpoints/ventas_limpias") \
    .trigger(availableNow=True) \
    .toTable(output_table)

# Iniciar el streaming
query.awaitTermination()


In [0]:
des

In [0]:
%python
query.exception()


In [0]:
%python
# Revisa el estado del streaming
query.status


In [0]:
select count(*) from my_catalog.my_schema.ventas_limpias;

In [0]:
describe history my_catalog.my_schema.ventas_limpias;

In [0]:
drop table if exists my_catalog.my_schema.ventas_limpias

In [0]:
%python
dbutils.fs.ls("/Volumes/my_catalog/my_schema/my_csv_python/")

In [0]:
describe history my_catalog.my_schema.my_csv_python;

# 6. Adding Column Metadata on Ingestion

Metadata: [Metadata Documentation](https://docs.databricks.com/aws/en/ingestion/file-metadata-column)

_metadata.file_modification_time: Adds the last modification time
_meatadata.file_name: Adds the input file name
current_timestamp(): Returns the current timesatamp

cast(form_unixtime(user_first_touch_timestamp /1000000) AS DATE)

In [0]:
select *, 
_metadata.file_name As source_file,
_metadata.file_modification_time As source_file_modification_time,
_metadata.file_size As source_file_size
from read_files(
  "/Volumes/my_catalog/my_schema/my_csv_python/",
  format => "csv"
)
limit 10;

### Creating the Final Bronze Table

In [0]:
drop table if exists my_catalog.my_schema.historical_users_bronze;

create table my_catalog.my_schema.historical_users_bronze
as
select *, 
_metadata.file_name As source_file,
_metadata.file_modification_time As source_file_modification_time,
_metadata.file_size As source_file_size
from read_files(
  "/Volumes/my_catalog/my_schema/my_csv_python/",
  format => "csv"
);

-- View the final bronze table
select * from my_catalog.my_schema.historical_users_bronze;

## Python


In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import LongType, TimestampType

# Drop the table if it exists
spark.sql("DROP TABLE IF EXISTS my_catalog.my_schema.historical_users_bronze")

# Read the CSV files and include metadata columns
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/my_catalog/my_schema/my_csv_python/") \
    .withColumn("source_file", lit("_metadata.file_path")) \
    .withColumn("source_file_modification_time", lit(None).cast(TimestampType())) \
    .withColumn("source_file_size", lit(None).cast(LongType())) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Write the DataFrame to a new table
df.write.saveAsTable("my_catalog.my_schema.historical_users_bronze")

# View the final bronze table
display(spark.sql("SELECT * FROM my_catalog.my_schema.historical_users_bronze"))